In [1]:
# bird_species_classifier.py

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from tensorflow.keras.preprocessing import image
import os

In [3]:
# Part 1 - Data Preprocessing

train_datagen = ImageDataGenerator(rescale=1./255,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

In [4]:
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size=(64, 64),
                                                 batch_size=32,
                                                 class_mode='categorical')

Found 150 images belonging to 16 classes.


In [5]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size=(64, 64),
                                            batch_size=32,
                                            class_mode='categorical')

Found 157 images belonging to 16 classes.


In [6]:
# Part 2 - Building the CNN

cnn = tf.keras.models.Sequential()

In [7]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=(64, 64, 3)))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

C:\Users\PMLS\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

In [9]:
cnn.add(tf.keras.layers.Flatten())

In [10]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn.add(tf.keras.layers.Dense(units=16, activation='softmax'))  # 16 classes for 16 bird species

In [11]:
cnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [12]:
cnn.fit(x=training_set, validation_data=test_set, epochs=25)

C:\Users\PMLS\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 200s 47s/step - accuracy: 0.0879 - loss: 2.7898 - val_accuracy: 0.1975 - val_loss: 2.6784
Epoch 2/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 154s 36s/step - accuracy: 0.1984 - loss: 2.5868 - val_accuracy: 0.1783 - val_loss: 2.6183
Epoch 3/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 131s 29s/step - accuracy: 0.1866 - loss: 2.4686 - val_accuracy: 0.2102 - val_loss: 2.6085
Epoch 4/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 127s 30s/step - accuracy: 0.2383 - loss: 2.3723 - val_accuracy: 0.2102 - val_loss: 2.5914
Epoch 5/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 132s 30s/step - accuracy: 0.2454 - loss: 2.2430 - val_accuracy: 0.2229 - val_loss: 2.5719
Epoch 6/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 131s 30s/step - accuracy: 0.2549 - loss: 2.2364 - val_accuracy: 0.2611 - val_loss: 2.5601
Epoch 7/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 126s 28s/step - accuracy: 0.3528 - loss: 2.1462 - val_accuracy: 0.2229 - val_loss: 2.6212
Epoch 8/25
5/5 ━━━━━━━━━━━━━━━━━━━━ 127s 30s/step - accuracy: 0.3436 - loss: 2.0965 - val_accuracy: 0.2866 - val_loss:

In [17]:
# Class labels (order must match model training)
bird_classes = [
    'Black Kite', 'black-winged stilt', 'Bonellis eagle', 'Chestnut-bellied rock thrush', 'Common myna', 'Grey tit', 'Hill pigeon', 'Himalayan bulbul',
    'Himalayan griffon vulture', 'House Sparrow', 'Indian Vulture', 'jungle owlet', 'large-billed crow', 'magpie-robin', 'Red-billed Blue Magpie', 'White-capped Redstart'
]

# Load and preprocess image
test_image = image.load_img('dataset/test_set/White-capped Redstart/100_4466.jpg', target_size=(64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis=0)
test_image = test_image / 255.0

# Predict
prediction = cnn.predict(test_image)
predicted_class = bird_classes[np.argmax(prediction)]

print(f"Predicted Bird Species: {predicted_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 643ms/step
Predicted Bird Species: Himalayan griffon vulture


In [15]:
# Save model

cnn.save('model/bird_species.h5')